<a href="https://colab.research.google.com/github/MrViincciLeRoy/Bot/blob/main/JellyBlog.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:


import requests
from bs4 import BeautifulSoup
import time
import re
from urllib.parse import urljoin, urlparse, quote_plus
from typing import List, Dict, Optional, Set
import json
from dataclasses import dataclass
from datetime import datetime
import logging

# Configure logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

@dataclass
class Source:
    text: str
    metadata: Dict
    url: str
    title: str
    reliability_score: float

class EnhancedPlantSpider:
    def __init__(self, delay: float = 1.5, max_sources: int = 10, articles_per_search: int = 3):
        """
        Initialize the Enhanced Plant Spider for RAG system

        Args:
            delay: Delay between requests in seconds
            max_sources: Maximum number of sources to collect
            articles_per_search: Number of articles to extract from each search page
        """
        self.delay = delay
        self.max_sources = max_sources
        self.articles_per_search = articles_per_search
        self.session = requests.Session()
        self.session.headers.update({
            'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
        })

        # Trusted plant information sources with search patterns
        self.search_sources = {
            'thespruce.com': {
                'reliability': 0.75,
                'search_url': 'https://www.thespruce.com/search?q={query}',
                'article_selector': 'a[href*="/plants/"], a[href*="/gardening/"]',
                'title_selector': '.card-title, .comp.card-list__title',
            },
            'extension.wisc.edu': {
                'reliability': 0.85,
                'search_url': 'https://hort.extension.wisc.edu/articles/?s={query}',
                'article_selector': 'a[href*="/articles/"]',
                'title_selector': '.entry-title, h2',
            },
            'ces.ncsu.edu': {
                'reliability': 0.85,
                'search_url': 'https://plants.ces.ncsu.edu/find_a_plant/?q={query}',
                'article_selector': 'a[href*="/plants/"]',
                'title_selector': 'h1, .plant-name',
                'browse_mode': True  # This site works better with browsing than search
            },
            'britannica.com': {
                'reliability': 0.9,
                'search_url': 'https://www.britannica.com/search?query={query}',
                'article_selector': 'a[href*="/plant/"], a[href*="/topic/"]',
                'title_selector': '.title, h3',
            },
            'rhs.org.uk': {
                'reliability': 0.9,
                'search_url': 'https://www.rhs.org.uk/search?query={query}',
                'article_selector': 'a[href*="/plants/"]',
                'title_selector': '.plant-header__title, h2',
            },
            'extension.umn.edu': {
                'reliability': 0.85,
                'search_url': 'https://extension.umn.edu/search?q={query}',
                'article_selector': 'a[href*="/plants"], a[href*="/garden"]',
                'title_selector': '.search-result-title, h3',
            }
        }

        # Direct reliable sources (non-search)
        self.direct_sources = {
            'en.wikipedia.org': {'reliability': 0.95, 'weight': 1.5},
            'kew.org': {'reliability': 0.95, 'weight': 1.5},
            'powo.science.kew.org': {'reliability': 0.95, 'weight': 1.5},
            'missouribotanicalgarden.org': {'reliability': 0.9, 'weight': 1.3},
        }

    def get_search_urls_for_plant(self, plant_name: str) -> List[str]:
        """
        Generate search URLs for a plant across different botanical sites
        """
        search_urls = []
        query = quote_plus(plant_name)

        # Add search URLs from known sources
        for domain, config in self.search_sources.items():
            search_url = config['search_url'].format(query=query)
            search_urls.append(search_url)

        # Add direct Wikipedia attempts (these often work)
        wiki_variations = [
            f"https://en.wikipedia.org/wiki/{plant_name.replace(' ', '_')}",
        ]

        # Add genus page if binomial name
        genus_species = plant_name.split()
        if len(genus_species) >= 2:
            genus = genus_species[0]
            wiki_variations.append(f"https://en.wikipedia.org/wiki/{genus}")

        search_urls.extend(wiki_variations)

        return search_urls

    def extract_article_links_from_search(self, search_url: str, plant_name: str) -> List[Dict[str, str]]:
        """
        Extract article links from search result pages

        Returns:
            List of dicts with 'url' and 'title' keys
        """
        try:
            logger.info(f"Processing search page: {search_url}")
            response = self.session.get(search_url, timeout=15)
            response.raise_for_status()

            soup = BeautifulSoup(response.content, 'html.parser')
            domain = urlparse(search_url).netloc

            # Remove unwanted elements
            for element in soup(['script', 'style', 'nav', 'header', 'footer']):
                element.decompose()

            articles = []

            # Handle different site structures
            if 'thespruce.com' in domain:
                articles = self._extract_thespruce_links(soup, search_url, plant_name)
            elif 'extension.wisc.edu' in domain:
                articles = self._extract_extension_links(soup, search_url, plant_name)
            elif 'ces.ncsu.edu' in domain:
                articles = self._extract_ncsu_links(soup, search_url, plant_name)
            elif 'britannica.com' in domain:
                articles = self._extract_britannica_links(soup, search_url, plant_name)
            elif 'rhs.org.uk' in domain:
                articles = self._extract_rhs_links(soup, search_url, plant_name)
            else:
                # Generic extraction
                articles = self._extract_generic_search_links(soup, search_url, plant_name)

            # Filter and rank articles
            filtered_articles = self._filter_relevant_articles(articles, plant_name)

            logger.info(f"Found {len(filtered_articles)} relevant articles from {domain}")
            return filtered_articles[:self.articles_per_search]

        except Exception as e:
            logger.error(f"Error processing search page {search_url}: {str(e)}")
            return []

    def _extract_thespruce_links(self, soup: BeautifulSoup, base_url: str, plant_name: str) -> List[Dict[str, str]]:
        """Extract links from The Spruce search results"""
        articles = []

        # The Spruce uses different selectors for search results
        selectors = [
            'a[href*="/plants/"]',
            'a[href*="/gardening/"]',
            '.comp.card-list__item a',
            '.search-results a[href*="thespruce.com"]'
        ]

        for selector in selectors:
            links = soup.select(selector)
            for link in links[:10]:  # Limit to avoid too many
                href = link.get('href')
                if href and not href.startswith('javascript:'):
                    full_url = urljoin(base_url, href)
                    title = link.get_text(strip=True)

                    # Get title from nearby elements if link text is not descriptive
                    if not title or len(title) < 10:
                        parent = link.find_parent()
                        if parent:
                            title_elem = parent.find(['h2', 'h3', '.title', '.card-title'])
                            if title_elem:
                                title = title_elem.get_text(strip=True)

                    if title and href:
                        articles.append({'url': full_url, 'title': title})

            if articles:  # If we found articles with this selector, stop trying others
                break

        return articles

    def _extract_extension_links(self, soup: BeautifulSoup, base_url: str, plant_name: str) -> List[Dict[str, str]]:
        """Extract links from extension site search results"""
        articles = []

        # Extension sites often have article lists
        selectors = [
            'a[href*="/articles/"]',
            'a[href*="/plants/"]',
            '.search-result a',
            '.entry-title a',
            'h2 a, h3 a'
        ]

        for selector in selectors:
            links = soup.select(selector)
            for link in links[:8]:
                href = link.get('href')
                title = link.get_text(strip=True)

                if href and title:
                    full_url = urljoin(base_url, href)
                    articles.append({'url': full_url, 'title': title})

            if articles:
                break

        return articles

    def _extract_ncsu_links(self, soup: BeautifulSoup, base_url: str, plant_name: str) -> List[Dict[str, str]]:
        """Extract links from NC State Extension plant database"""
        articles = []

        # NCSU plant database has specific structure
        selectors = [
            'a[href*="/plants/"]',
            '.plant-list a',
            '.search-results a',
            'td a',  # Sometimes in tables
            '.plant-name a'
        ]

        for selector in selectors:
            links = soup.select(selector)
            for link in links[:10]:
                href = link.get('href')
                title = link.get_text(strip=True)

                if href and title and len(title) > 3:
                    full_url = urljoin(base_url, href)
                    articles.append({'url': full_url, 'title': title})

        return articles

    def _extract_britannica_links(self, soup: BeautifulSoup, base_url: str, plant_name: str) -> List[Dict[str, str]]:
        """Extract links from Britannica search results"""
        articles = []

        # Britannica search result selectors
        selectors = [
            'a[href*="/plant/"]',
            'a[href*="/topic/"]',
            '.search-results a',
            '.title a',
            'h3 a'
        ]

        for selector in selectors:
            links = soup.select(selector)
            for link in links[:6]:
                href = link.get('href')
                title = link.get_text(strip=True)

                if href and title:
                    full_url = urljoin('https://www.britannica.com', href)
                    articles.append({'url': full_url, 'title': title})

            if articles:
                break

        return articles

    def _extract_rhs_links(self, soup: BeautifulSoup, base_url: str, plant_name: str) -> List[Dict[str, str]]:
        """Extract links from RHS search results"""
        articles = []

        selectors = [
            'a[href*="/plants/"]',
            '.search-result a',
            '.plant-search-result a',
            'h2 a, h3 a'
        ]

        for selector in selectors:
            links = soup.select(selector)
            for link in links[:8]:
                href = link.get('href')
                title = link.get_text(strip=True)

                if href and title:
                    full_url = urljoin(base_url, href)
                    articles.append({'url': full_url, 'title': title})

        return articles

    def _extract_generic_search_links(self, soup: BeautifulSoup, base_url: str, plant_name: str) -> List[Dict[str, str]]:
        """Generic extraction for unknown sites"""
        articles = []

        # Generic selectors that might work on various sites
        selectors = [
            'a[href*="plant"]',
            '.search-result a',
            '.result a',
            'h2 a, h3 a',
            '.title a',
            '.entry-title a'
        ]

        for selector in selectors:
            links = soup.select(selector)[:15]  # Limit to avoid noise
            for link in links:
                href = link.get('href')
                title = link.get_text(strip=True)

                if href and title and len(title) > 5:
                    # Skip obvious non-content links
                    if not any(skip in href.lower() for skip in ['contact', 'about', 'privacy', 'terms']):
                        full_url = urljoin(base_url, href)
                        articles.append({'url': full_url, 'title': title})

        return articles

    def _filter_relevant_articles(self, articles: List[Dict[str, str]], plant_name: str) -> List[Dict[str, str]]:
        """Filter and rank articles by relevance to the plant"""
        if not articles:
            return []

        plant_terms = plant_name.lower().split()
        genus = plant_terms[0] if plant_terms else ""

        scored_articles = []
        seen_urls = set()

        for article in articles:
            url = article['url']
            title = article['title'].lower()

            # Skip duplicates
            if url in seen_urls:
                continue
            seen_urls.add(url)

            # Calculate relevance score
            score = 0

            # Exact plant name match (highest score)
            if plant_name.lower() in title:
                score += 10

            # Genus match
            if genus and genus in title:
                score += 5

            # Individual term matches
            for term in plant_terms:
                if term in title:
                    score += 2

            # Plant-related keywords
            plant_keywords = ['plant', 'flower', 'tree', 'shrub', 'garden', 'cultivation', 'growing']
            for keyword in plant_keywords:
                if keyword in title:
                    score += 1

            # Prefer specific plant pages over general garden advice
            if any(specific in url.lower() for specific in ['/plants/', '/plant/', '/species/']):
                score += 3

            scored_articles.append((score, article))

        # Sort by score (highest first) and return articles
        scored_articles.sort(key=lambda x: x[0], reverse=True)
        return [article for score, article in scored_articles if score > 0]

    def extract_plant_info(self, url: str) -> Optional[Source]:
        """
        Enhanced plant information extraction with better content detection
        """
        try:
            response = self.session.get(url, timeout=15)
            response.raise_for_status()

            soup = BeautifulSoup(response.content, 'html.parser')

            # Remove unwanted elements
            for element in soup(['script', 'style', 'nav', 'header', 'footer', 'aside', '.ad', '.advertisement']):
                element.decompose()

            # Extract title
            title = self._extract_title(soup, url)

            # Extract main content
            content = self._extract_content(soup, url)

            if not content or len(content.strip()) < 100:
                logger.debug(f"Insufficient content from {url} (length: {len(content) if content else 0})")
                return None

            # Calculate reliability score
            domain = urlparse(url).netloc
            reliability_score = self._calculate_reliability(domain, content)

            # Create metadata optimized for RAG system
            metadata = {
                'source': self._get_source_name(domain, title),
                'reliability': self._get_reliability_level(reliability_score),
                'url': url,
                'domain': domain,
                'title': title,
                'scraped_date': datetime.now().strftime('%Y-%m-%d'),
                'content_type': self._classify_content_type(content, url)
            }

            return Source(
                text=content,
                metadata=metadata,
                url=url,
                title=title,
                reliability_score=reliability_score
            )

        except Exception as e:
            logger.error(f"Error extracting from {url}: {str(e)}")
            return None

    def _extract_title(self, soup: BeautifulSoup, url: str) -> str:
        """Enhanced title extraction"""
        # Try different title selectors in order of preference
        selectors = [
            'h1.plant-name',
            'h1.entry-title',
            'h1.title',
            '.plant-header__title',
            '.plant-title',
            'h1',
            'title'
        ]

        for selector in selectors:
            elem = soup.select_one(selector)
            if elem:
                title = elem.get_text(strip=True)
                if title and len(title) > 3:
                    return title

        return "Unknown Plant"

    def _extract_content(self, soup: BeautifulSoup, url: str) -> str:
        """Enhanced content extraction with site-specific handling"""
        domain = urlparse(url).netloc

        if 'wikipedia.org' in domain:
            return self._extract_wikipedia_content(soup)
        elif 'thespruce.com' in domain:
            return self._extract_thespruce_content(soup)
        elif 'extension.wisc.edu' in domain or 'extension.' in domain:
            return self._extract_extension_content(soup)
        elif 'ces.ncsu.edu' in domain:
            return self._extract_ncsu_content(soup)
        elif 'britannica.com' in domain:
            return self._extract_britannica_content(soup)
        elif 'rhs.org.uk' in domain:
            return self._extract_rhs_content(soup)
        else:
            return self._extract_generic_content(soup)

    def _extract_thespruce_content(self, soup: BeautifulSoup) -> str:
        """Extract content from The Spruce articles"""
        content_parts = []

        # The Spruce article structure
        selectors = [
            '.comp.mntl-sc-block-html',
            '.comp.article-content p',
            '.content-block p',
            'article p',
            '.entry-content p'
        ]

        for selector in selectors:
            elements = soup.select(selector)
            if elements:
                for elem in elements[:8]:
                    text = elem.get_text(strip=True)
                    if text and len(text) > 30 and self._is_content_text(text):
                        content_parts.append(text)
                if len(content_parts) >= 3:
                    break

        return "\n\n".join(content_parts)

    def _extract_extension_content(self, soup: BeautifulSoup) -> str:
        """Extract content from extension articles"""
        content_parts = []

        selectors = [
            '.entry-content p',
            '.article-content p',
            '.content p',
            'main p',
            '#content p'
        ]

        for selector in selectors:
            elements = soup.select(selector)
            if elements:
                for elem in elements[:10]:
                    text = elem.get_text(strip=True)
                    if text and len(text) > 40 and self._is_content_text(text):
                        content_parts.append(text)
                if content_parts:
                    break

        return "\n\n".join(content_parts)

    def _extract_ncsu_content(self, soup: BeautifulSoup) -> str:
        """Extract content from NC State plant database"""
        content_parts = []

        # NCSU specific selectors
        selectors = [
            '.plant-description p',
            '.plant-details p',
            '.plant-info p',
            'td',  # Plant details often in tables
            '.content p',
            'p'
        ]

        for selector in selectors:
            elements = soup.select(selector)
            if elements:
                for elem in elements[:12]:
                    text = elem.get_text(strip=True)
                    if text and len(text) > 25 and self._is_content_text(text):
                        content_parts.append(text)
                if len(content_parts) >= 4:
                    break

        return "\n\n".join(content_parts)

    def _extract_wikipedia_content(self, soup: BeautifulSoup) -> str:
        """Extract content from Wikipedia pages"""
        content_parts = []

        content_div = soup.find('div', {'id': 'mw-content-text'})
        if content_div:
            paragraphs = content_div.find_all('p', recursive=True)[:10]

            for p in paragraphs:
                text = p.get_text(strip=True)
                if text and len(text) > 50:
                    content_parts.append(text)

        return "\n\n".join(content_parts)

    def _extract_britannica_content(self, soup: BeautifulSoup) -> str:
        """Extract content from Britannica articles"""
        content_parts = []

        article = soup.find('article') or soup.find('div', class_='article-content')
        if article:
            paragraphs = article.find_all('p')[:8]
            for p in paragraphs:
                text = p.get_text(strip=True)
                if text and len(text) > 50 and self._is_content_text(text):
                    content_parts.append(text)

        return "\n\n".join(content_parts)

    def _extract_rhs_content(self, soup: BeautifulSoup) -> str:
        """Extract content from RHS articles"""
        content_parts = []

        selectors = [
            '.plant-description p',
            '.plant-summary p',
            '.content-area p',
            '.main-content p',
            'article p'
        ]

        for selector in selectors:
            elements = soup.select(selector)
            if elements:
                for elem in elements[:6]:
                    text = elem.get_text(strip=True)
                    if text and len(text) > 50 and self._is_content_text(text):
                        content_parts.append(text)
                if content_parts:
                    break

        return "\n\n".join(content_parts)

    def _extract_generic_content(self, soup: BeautifulSoup) -> str:
        """Enhanced generic content extraction"""
        content_parts = []

        # Try content-specific selectors first
        priority_selectors = [
            'article p',
            '.entry-content p',
            '.post-content p',
            '.content p',
            '.main-content p',
            '#content p'
        ]

        for selector in priority_selectors:
            elements = soup.select(selector)
            if elements:
                for elem in elements[:10]:
                    text = elem.get_text(strip=True)
                    if text and len(text) > 40 and self._is_content_text(text):
                        content_parts.append(text)
                if len(content_parts) >= 3:
                    break

        # Fallback to all paragraphs
        if not content_parts:
            paragraphs = soup.find_all('p')
            for p in paragraphs[:20]:
                text = p.get_text(strip=True)
                if text and len(text) > 50 and self._is_content_text(text):
                    content_parts.append(text)
                if len(content_parts) >= 5:
                    break

        return "\n\n".join(content_parts[:8])

    def _is_content_text(self, text: str) -> bool:
        """Check if text is actual content vs navigation/ads"""
        text_lower = text.lower()

        # Skip navigation, ads, etc.
        skip_phrases = [
            'cookie', 'privacy', 'subscribe', 'newsletter', 'advertisement',
            'menu', 'navigation', 'share this', 'follow us', 'contact us',
            'terms of service', 'all rights reserved'
        ]

        return not any(phrase in text_lower for phrase in skip_phrases)

    def _calculate_reliability(self, domain: str, content: str) -> float:
        """Calculate reliability score"""
        base_score = 0.5

        # Check direct sources
        if domain in self.direct_sources:
            base_score = self.direct_sources[domain]['reliability']
        else:
            # Check search sources
            for search_domain, config in self.search_sources.items():
                if search_domain in domain:
                    base_score = config['reliability']
                    break

        # Content quality indicators
        content_lower = content.lower()

        if any(term in content_lower for term in ['scientific name', 'botanical', 'taxonomy']):
            base_score += 0.1

        if len(content) > 1000:
            base_score += 0.05

        return min(1.0, base_score)

    def _get_reliability_level(self, score: float) -> str:
        """Convert reliability score to level"""
        if score >= 0.9:
            return "very_high"
        elif score >= 0.8:
            return "high"
        elif score >= 0.7:
            return "medium"
        else:
            return "low"

    def collect_plant_sources(self, plant_name: str) -> List[Dict]:
        """
        Main method to collect sources using enhanced search-based approach
        """
        logger.info(f"Starting enhanced collection for: {plant_name}")

        search_urls = self.get_search_urls_for_plant(plant_name)

        all_article_links = []
        sources = []
        processed_urls = set()
        processed_domains = set()

        # Step 1: Extract article links from search pages
        for search_url in search_urls:
            if len(all_article_links) >= self.max_sources * 2:  # Get extra links to filter from
                break

            article_links = self.extract_article_links_from_search(search_url, plant_name)
            all_article_links.extend(article_links)

            time.sleep(self.delay)

        logger.info(f"Found {len(all_article_links)} total article links")

        # Step 2: Process article links to extract content
        for article in all_article_links:
            if len(sources) >= self.max_sources:
                break

            url = article['url']

            # Skip if already processed
            if url in processed_urls:
                continue

            # Limit to 2 articles per domain for diversity
            domain = urlparse(url).netloc
            domain_count = sum(1 for s in sources if urlparse(s['metadata']['url']).netloc == domain)
            if domain_count >= 2:
                continue

            processed_urls.add(url)

            logger.info(f"Processing article: {article['title'][:50]}...")

            try:
                source = self.extract_plant_info(url)
                if source and len(source.text.strip()) > 150:
                    # Create RAG-optimized source format
                    rag_source = {
                        "text": source.text,
                        "metadata": source.metadata
                    }
                    sources.append(rag_source)
                    logger.info(f"✓ Extracted content from {domain}")
                else:
                    logger.debug(f"✗ Insufficient content from {url}")

            except Exception as e:
                logger.debug(f"✗ Error processing {url}: {str(e)}")

            time.sleep(self.delay)

        # Sort by reliability for RAG system
        sources.sort(key=lambda x: self._get_rag_sort_score(x['metadata']), reverse=True)

        logger.info(f"Successfully collected {len(sources)} sources for {plant_name}")
        return sources

    def _get_source_name(self, domain: str, title: str) -> str:
        """Get a clean source name for RAG metadata"""
        source_names = {
            'en.wikipedia.org': 'Wikipedia',
            'www.britannica.com': 'Encyclopædia Britannica',
            'www.thespruce.com': 'The Spruce',
            'hort.extension.wisc.edu': 'Wisconsin Horticulture Extension',
            'plants.ces.ncsu.edu': 'NC State Extension Plants',
            'extension.umn.edu': 'University of Minnesota Extension',
            'www.rhs.org.uk': 'Royal Horticultural Society',
            'www.kew.org': 'Royal Botanic Gardens, Kew',
            'powo.science.kew.org': 'Plants of the World Online',
            'www.missouribotanicalgarden.org': 'Missouri Botanical Garden',
            'plants.usda.gov': 'USDA Plants Database',
            'plantnet.rbgsyd.nsw.gov.au': 'PlantNET'
        }

        return source_names.get(domain, title.split(' - ')[0] if ' - ' in title else domain.replace('www.', '').title())

    def _classify_content_type(self, content: str, url: str) -> str:
        """Classify the type of content for RAG context"""
        content_lower = content.lower()

        if any(term in content_lower for term in ['scientific name', 'botanical', 'taxonomy', 'genus', 'species']):
            return 'botanical_reference'
        elif any(term in content_lower for term in ['growing', 'planting', 'cultivation', 'care', 'garden']):
            return 'cultivation_guide'
        elif any(term in content_lower for term in ['native', 'habitat', 'distribution', 'ecology']):
            return 'ecological_information'
        elif any(term in content_lower for term in ['description', 'appearance', 'characteristics', 'features']):
            return 'plant_description'
        else:
            return 'general_information'

    def _get_rag_sort_score(self, metadata: Dict) -> float:
        """Get sorting score optimized for RAG system"""
        reliability_scores = {
            'very_high': 1.0,
            'high': 0.8,
            'medium': 0.6,
            'low': 0.4
        }

        base_score = reliability_scores.get(metadata.get('reliability', 'low'), 0.4)

        # Bonus for specific content types that are more informative
        content_type_bonus = {
            'botanical_reference': 0.2,
            'plant_description': 0.15,
            'cultivation_guide': 0.1,
            'ecological_information': 0.1,
            'general_information': 0.0
        }

        base_score += content_type_bonus.get(metadata.get('content_type', 'general_information'), 0.0)

        return min(1.0, base_score)

    def save_sources_for_rag(self, sources: List[Dict], filename: str, plant_name: str):
        """Save sources in RAG-optimized format with additional context"""
        rag_data = {
            "plant_name": plant_name,
            "collection_date": datetime.now().strftime('%Y-%m-%d'),
            "total_sources": len(sources),
            "sources": sources
        }

        with open(filename, 'w', encoding='utf-8') as f:
            json.dump(rag_data, f, indent=2, ensure_ascii=False)

        # Also save just the sources array for direct RAG use
        sources_only_filename = filename.replace('.json', '_sources_only.json')
        with open(sources_only_filename, 'w', encoding='utf-8') as f:
            json.dump(sources, f, indent=2, ensure_ascii=False)

# Example usage
def main(name=None):
    # Initialize enhanced spider
    spider = EnhancedPlantSpider(delay=1.5, max_sources=8, articles_per_search=3)

    # Test with Rosa rubiginosa
    plant_name = name if name else "Rosa rubiginosa"
    sources = spider.collect_plant_sources(plant_name)

    # Save results
    filename = f"{plant_name.replace(' ', '_')}_enhanced_sources.json"
    spider.save_sources_for_rag(sources, filename, plant_name)

    # Print detailed summary
    print(f"\n{'='*60}")
    print(f"ENHANCED PLANT SPIDER RESULTS FOR: {plant_name}")
    print(f"{'='*60}")
    print(f"Total sources collected: {len(sources)}")
    print()

    for i, source in enumerate(sources, 1):
        metadata = source['metadata']
        print(f"{i}. {metadata['source']}")
        print(f"   Title: {metadata['title']}")
        print(f"   Reliability: {metadata['reliability']}")
        print(f"   Content length: {metadata['content_type']} characters")
        print(f"   URL: {metadata.keys()}")
    return sources

In [ ]:


!pip install faiss-cpu  # For CPU
# OR
!pip install faiss-gpu  # For GPU acceleration

!pip install sentence-transformers beautifulsoup4 requests

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.4/31.4 MB 37.5 MB/s eta 0:00:00
ERROR: Could not find a version that satisfies the requirement faiss-gpu (from versions: none)
ERROR: No matching distribution found for faiss-gpu


In [ ]:
data = main()


ENHANCED PLANT SPIDER RESULTS FOR: Rosa rubiginosa
Total sources collected: 6

1. NC State Extension Plants
   Title: Rosa America™'JACclam'
   Reliability: very_high
   Content length: cultivation_guide characters
   URL: dict_keys(['source', 'reliability', 'url', 'domain', 'title', 'scraped_date', 'content_type'])
2. NC State Extension Plants
   Title: Hibiscus rosa-sinensis
   Reliability: very_high
   Content length: botanical_reference characters
   URL: dict_keys(['source', 'reliability', 'url', 'domain', 'title', 'scraped_date', 'content_type'])
3. Encyclopædia Britannica
   Title: shrub rose
   Reliability: very_high
   Content length: general_information characters
   URL: dict_keys(['source', 'reliability', 'url', 'domain', 'title', 'scraped_date', 'content_type'])
4. Encyclopædia Britannica
   Title: sweetbrier
   Reliability: very_high
   Content length: ecological_information characters
   URL: dict_keys(['source', 'reliability', 'url', 'domain', 'title', 'scraped_date', 

In [ ]:

from sentence_transformers import SentenceTransformer
import numpy as np

# Load embedding model
model = SentenceTransformer('all-MiniLM-L6-v2')

# Extract texts
texts = [item['text'] for item in data]

# Generate embeddings
embeddings = model.encode(texts)  # Shape: (n_texts, embedding_dim)
print("Embedding shape:", embeddings.shape)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

NameError: name 'data' is not defined

In [ ]:

import faiss

# Get embedding dimension
d = embeddings.shape[1]

# Create a FAISS index (flat L2 distance)
index = faiss.IndexFlatL2(d)

# Add vectors to the index
index.add(np.array(embeddings).astype('float32'))

print(f"Number of vectors in index: {index.ntotal}")

NameError: name 'embeddings' is not defined

In [ ]:
metadata = [item['metadata'] for item in data]

In [ ]:

"""
Streamlined FAISS RAG System for Pre-Collected Plant Data
Works with your existing spider output format
"""

import json
import pickle
from typing import List, Dict, Optional
from pathlib import Path
import logging
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np

# Configure logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)


class PlantRAGSystem:
    """
    Lightweight FAISS-based RAG system
    Works directly with your spider's output format
    """

    def __init__(self, db_path: str = "./plant_knowledge_db"):
        """Initialize FAISS RAG system"""
        self.db_path = Path(db_path)
        self.db_path.mkdir(exist_ok=True)

        # Initialize embedding model
        logger.info("Loading embedding model...")
        self.embedding_model = SentenceTransformer('all-MiniLM-L6-v2')
        self.embedding_dim = 384

        # Storage
        self.index = None
        self.documents = []  # Text chunks
        self.metadatas = []  # Metadata for each chunk

        # File paths
        self.index_path = self.db_path / "faiss_index.bin"
        self.docs_path = self.db_path / "documents.pkl"
        self.meta_path = self.db_path / "metadata.pkl"

        # Load existing database
        self._load_database()

        logger.info(f"RAG initialized. Current items: {len(self.documents)}")

    def _initialize_index(self):
        """Initialize FAISS index"""
        self.index = faiss.IndexFlatL2(self.embedding_dim)

    def _load_database(self):
        """Load existing database from disk"""
        if self.index_path.exists() and self.docs_path.exists():
            try:
                self.index = faiss.read_index(str(self.index_path))

                with open(self.docs_path, 'rb') as f:
                    self.documents = pickle.load(f)

                with open(self.meta_path, 'rb') as f:
                    self.metadatas = pickle.load(f)

                logger.info(f"✓ Loaded existing database: {len(self.documents)} chunks")
            except Exception as e:
                logger.warning(f"Could not load database: {e}. Starting fresh.")
                self._initialize_index()
        else:
            self._initialize_index()

    def _save_database(self):
        """Save database to disk"""
        try:
            faiss.write_index(self.index, str(self.index_path))

            with open(self.docs_path, 'wb') as f:
                pickle.dump(self.documents, f)

            with open(self.meta_path, 'wb') as f:
                pickle.dump(self.metadatas, f)

            logger.info("✓ Database saved")
        except Exception as e:
            logger.error(f"Error saving database: {e}")

    def chunk_text(self, text: str, chunk_size: int = 400, overlap: int = 50) -> List[str]:
        """Split text into overlapping chunks"""
        words = text.split()
        chunks = []

        for i in range(0, len(words), chunk_size - overlap):
            chunk = ' '.join(words[i:i + chunk_size])
            if len(chunk.strip()) > 50:  # Minimum chunk size
                chunks.append(chunk)

        return chunks

    def add_plant(self, plant_name: str, spider_output: List[Dict]) -> int:
        """
        Add plant to knowledge base

        Args:
            plant_name: Name of the plant (e.g., "Rosa rubiginosa")
            spider_output: List of dicts with 'text' and 'metadata' keys

        Returns:
            Number of chunks added
        """

        if not spider_output:
            logger.warning(f"No data provided for {plant_name}")

            spider_results = main(plant_name)
            spider_output = spider_results
            #return 0

        total_chunks = 0
        new_embeddings = []
        new_documents = []
        new_metadatas = []

        for source in spider_output:
            text_content = source['text']
            metadata = source['metadata']

            # Add plant name to metadata
            metadata['plant_name'] = plant_name

            # Chunk the text
            chunks = self.chunk_text(text_content)

            if not chunks:
                continue

            # Generate embeddings
            logger.info(f"  Embedding {len(chunks)} chunks from {metadata.get('source', 'Unknown')}")
            embeddings = self.embedding_model.encode(
                chunks,
                show_progress_bar=False,
                convert_to_numpy=True
            )

            # Store each chunk
            for i, (chunk, embedding) in enumerate(zip(chunks, embeddings)):
                chunk_meta = metadata.copy()
                chunk_meta['chunk_id'] = i
                chunk_meta['total_chunks'] = len(chunks)

                new_embeddings.append(embedding)
                new_documents.append(chunk)
                new_metadatas.append(chunk_meta)

            total_chunks += len(chunks)
            logger.info(f"  ✓ {len(chunks)} chunks from {metadata['source']}")

        # Add to FAISS index
        if new_embeddings:
            embeddings_array = np.array(new_embeddings).astype('float32')
            self.index.add(embeddings_array)
            self.documents.extend(new_documents)
            self.metadatas.extend(new_metadatas)

            # Save to disk
            self._save_database()

            logger.info(f"🎉 Total: {total_chunks} chunks added for {plant_name}")

        return total_chunks

    def search(
        self,
        query: str,
        n_results: int = 5,
        plant_name: Optional[str] = None,
        min_relevance: float = 0.0
    ) -> Dict:
        """
        Search the knowledge base

        Args:
            query: Search query
            n_results: Number of results to return
            plant_name: Filter by specific plant
            min_relevance: Minimum relevance threshold (0-1)

        Returns:
            Dict with documents, metadatas, and relevance scores
        """

        if len(self.documents) == 0:
            return {
                "documents": [],
                "metadatas": [],
                "relevance_scores": [],
                "distances": []
            }

        # Generate query embedding
        query_embedding = self.embedding_model.encode([query])[0].astype('float32')

        # Search (get extra results for filtering)
        search_k = min(n_results * 3, len(self.documents))
        distances, indices = self.index.search(
            np.array([query_embedding]),
            search_k
        )

        # Filter and collect results
        filtered_docs = []
        filtered_metas = []
        filtered_distances = []
        filtered_relevance = []

        for dist, idx in zip(distances[0], indices[0]):
            if idx >= len(self.metadatas):
                continue

            meta = self.metadatas[idx]

            # Apply plant filter
            if plant_name and meta.get('plant_name') != plant_name:
                continue

            # Calculate relevance (inverse of distance)
            relevance = 1 / (1 + dist)

            # Apply relevance threshold
            if relevance < min_relevance:
                continue

            filtered_docs.append(self.documents[idx])
            filtered_metas.append(meta)
            filtered_distances.append(float(dist))
            filtered_relevance.append(float(relevance))

            if len(filtered_docs) >= n_results:
                break

        return {
            "documents": filtered_docs,
            "metadatas": filtered_metas,
            "distances": filtered_distances,
            "relevance_scores": filtered_relevance
        }

    def build_prompt(self, query: str, search_results: Dict) -> Optional[str]:
        """Build LLM prompt with retrieved context"""

        if not search_results['documents']:
            return None

        # Build context sections
        context_parts = []
        for doc, meta, rel in zip(
            search_results['documents'],
            search_results['metadatas'],
            search_results['relevance_scores']
        ):
            reliability = meta.get('reliability', 'unknown').upper()
            source = meta.get('source', 'Unknown')
            title = meta.get('title', 'Untitled')
            plant = meta.get('plant_name', 'Unknown')

            context_parts.append(
                f"[{reliability}] {source} - {title} "
                f"(Plant: {plant}, Relevance: {rel:.2f}):\n{doc}"
            )

        context = "\n\n".join(context_parts)

        prompt = f"""You are a botanical expert. Answer based on these scientific sources:

SOURCES:
{context}

QUESTION: {query}

INSTRUCTIONS:
- Answer ONLY using information above
- Cite sources (e.g., "According to Britannica...")
- If sources conflict, mention both viewpoints
- If answer not in sources, say so clearly
- Prioritize higher reliability and relevance sources

ANSWER:"""

        return prompt

    def ask(
        self,
        query: str,
        plant_name: Optional[str] = None,
        n_results: int = 5,
        min_relevance: float = 0.3
    ) -> Dict:
        """
        Complete RAG query

        Returns:
            Dict with prompt, sources, and search results
        """

        # Search
        results = self.search(
            query,
            n_results=n_results,
            plant_name=plant_name,
            min_relevance=min_relevance
        )

        if not results['documents']:
            return {
                "prompt": None,
                "sources": [],
                "answer": f"No relevant information found (searched {len(self.documents)} chunks)",
                "results": results
            }

        # Build prompt
        prompt = self.build_prompt(query, results)

        # Extract unique sources
        sources = []
        seen = set()
        for meta in results['metadatas']:
            key = (meta.get('source'), meta.get('url'))
            if key not in seen:
                sources.append({
                    "plant": meta.get('plant_name', 'Unknown'),
                    "source": meta.get('source', 'Unknown'),
                    "title": meta.get('title', 'Untitled'),
                    "url": meta.get('url', ''),
                    "reliability": meta.get('reliability', 'unknown')
                })
                seen.add(key)

        return {
            "prompt": prompt,
            "sources": sources,
            "answer": None,  # Fill this with LLM response
            "results": results
        }

    def stats(self) -> Dict:
        """Get database statistics"""

        if len(self.documents) == 0:
            return {
                "total_chunks": 0,
                "unique_plants": 0,
                "plants": [],
                "unique_sources": 0,
                "sources": []
            }

        plants = set()
        sources = set()

        for meta in self.metadatas:
            plants.add(meta.get('plant_name', 'Unknown'))
            sources.add(meta.get('source', 'Unknown'))

        return {
            "total_chunks": len(self.documents),
            "unique_plants": len(plants),
            "plants": sorted(list(plants)),
            "unique_sources": len(sources),
            "sources": sorted(list(sources))
        }

    def get_plant_info(self, plant_name: str) -> Dict:
        """Get all stored info about a specific plant"""

        plant_chunks = []
        plant_sources = set()

        for doc, meta in zip(self.documents, self.metadatas):
            if meta.get('plant_name') == plant_name:
                plant_chunks.append({
                    "text": doc,
                    "source": meta.get('source'),
                    "title": meta.get('title')
                })
                plant_sources.add(meta.get('source'))

        return {
            "plant": plant_name,
            "total_chunks": len(plant_chunks),
            "sources": sorted(list(plant_sources)),
            "chunks": plant_chunks
        }


# ============================================
# USAGE EXAMPLES
# ============================================

def example_usage(plantname=None):
    """How to use with your spider output"""

    # Your spider output (from the document you provided)
    _spider_results = [
        {
            'text': 'sweetbrier, (Rosa eglanteria, orR. rubiginosa), small, prickly wildrose...',
            'metadata': {
                'source': 'Encyclopædia Britannica',
                'reliability': 'very_high',
                'url': 'https://www.britannica.com/plant/sweetbrier',
                'domain': 'www.britannica.com',
                'title': 'sweetbrier',
                'scraped_date': '2025-10-08',
                'content_type': 'ecological_information'
            }
        },
        # ... more sources
    ]
    spider_results = main(plantname)
    # Initialize RAG
    rag = PlantRAGSystem()

    # Add plant
    chunks_added = rag.add_plant("Rosa rubiginosa", spider_results)
    print(f"Added {chunks_added} chunks")

    # Ask questions
    result = rag.ask("What does Rosa rubiginosa look like?")

    if result['prompt']:
        print("\n=== PROMPT FOR LLM ===")
        print(result['prompt'])
        print("\n=== SOURCES ===")
        for src in result['sources']:
            print(f"- [{src['reliability']}] {src['source']}: {src['title']}")

    # Database stats
    stats = rag.stats()
    print(f"\nDatabase: {stats['total_chunks']} chunks, {stats['unique_plants']} plants")


def load_from_file_example():
    """Load spider results from JSON file"""

    # If you saved your spider output to JSON:
    with open('rosa_rubiginosa_results.json', 'r') as f:
        spider_data = json.load(f)

    rag = PlantRAGSystem()
    rag.add_plant("Rosa rubiginosa", spider_data)

    # Query
    result = rag.ask("Is this plant fragrant?", min_relevance=0.4)
    print(result['prompt'])


# ============================================
# QUICK START
# ============================================

if __name__ == "__main__":

    # Quick test
    rag = PlantRAGSystem()

    # Check stats
    stats = rag.stats()
    print(f"Database contains: {stats['total_chunks']} chunks")
    print(f"Plants: {stats['plants']}")

    # Example query
    if stats['total_chunks'] > 0:
        result = rag.ask("What does this plant look like?")
        if result['prompt']:
            print("\n" + "="*60)
            print(result['prompt'][:500] + "...")
            print("="*60)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Database contains: 0 chunks
Plants: []


In [ ]:

#from your_module import PlantRAGSystem

# Initialize
rag = PlantRAGSystem()

# Add your spider results directly
spider_output = [
    {'text': '...', 'metadata': {...}},
    {'text': '...', 'metadata': {...}},
    # ... (your format exactly)
]

# Add to database
rag.add_plant("Rosa rubiginosa", spider_output=None)

# Ask questions
result = rag.ask("What does Rosa rubiginosa look like?", min_relevance=0.4)

# Use the prompt with any LLM
if result['prompt']:
    print(result['prompt'])
    # Send to Claude, GPT, etc.


ENHANCED PLANT SPIDER RESULTS FOR: Rosa rubiginosa
Total sources collected: 6

1. NC State Extension Plants
   Title: Rosa America™'JACclam'
   Reliability: very_high
   Content length: cultivation_guide characters
   URL: dict_keys(['source', 'reliability', 'url', 'domain', 'title', 'scraped_date', 'content_type'])
2. NC State Extension Plants
   Title: Hibiscus rosa-sinensis
   Reliability: very_high
   Content length: botanical_reference characters
   URL: dict_keys(['source', 'reliability', 'url', 'domain', 'title', 'scraped_date', 'content_type'])
3. Encyclopædia Britannica
   Title: shrub rose
   Reliability: very_high
   Content length: general_information characters
   URL: dict_keys(['source', 'reliability', 'url', 'domain', 'title', 'scraped_date', 'content_type'])
4. Encyclopædia Britannica
   Title: sweetbrier
   Reliability: very_high
   Content length: ecological_information characters
   URL: dict_keys(['source', 'reliability', 'url', 'domain', 'title', 'scraped_date', 

In [ ]:

result = rag.ask(
    "What does Rosa rubiginosa look like?",
    #generate_answer=True
)
print(result['answer'])
# Gets basic extraction from sources

None


In [ ]:

"""
Streamlined FAISS RAG System for Pre-Collected Plant Data
Works with your existing spider output format
"""

import json
import pickle
from typing import List, Dict, Optional
from pathlib import Path
import logging
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np

# Configure logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)


class PlantRAGSystem:
    """
    Lightweight FAISS-based RAG system
    Works directly with your spider's output format
    """

    def __init__(self, db_path: str = "./plant_knowledge_db"):
        """Initialize FAISS RAG system"""
        self.db_path = Path(db_path)
        self.db_path.mkdir(exist_ok=True)

        # Initialize embedding model
        logger.info("Loading embedding model...")
        self.embedding_model = SentenceTransformer('all-MiniLM-L6-v2')
        self.embedding_dim = 384

        # Storage
        self.index = None
        self.documents = []  # Text chunks
        self.metadatas = []  # Metadata for each chunk

        # File paths
        self.index_path = self.db_path / "faiss_index.bin"
        self.docs_path = self.db_path / "documents.pkl"
        self.meta_path = self.db_path / "metadata.pkl"

        # Load existing database
        self._load_database()

        logger.info(f"RAG initialized. Current items: {len(self.documents)}")

    def _initialize_index(self):
        """Initialize FAISS index"""
        self.index = faiss.IndexFlatL2(self.embedding_dim)

    def _load_database(self):
        """Load existing database from disk"""
        if self.index_path.exists() and self.docs_path.exists():
            try:
                self.index = faiss.read_index(str(self.index_path))

                with open(self.docs_path, 'rb') as f:
                    self.documents = pickle.load(f)

                with open(self.meta_path, 'rb') as f:
                    self.metadatas = pickle.load(f)

                logger.info(f"✓ Loaded existing database: {len(self.documents)} chunks")
            except Exception as e:
                logger.warning(f"Could not load database: {e}. Starting fresh.")
                self._initialize_index()
        else:
            self._initialize_index()

    def _save_database(self):
        """Save database to disk"""
        try:
            faiss.write_index(self.index, str(self.index_path))

            with open(self.docs_path, 'wb') as f:
                pickle.dump(self.documents, f)

            with open(self.meta_path, 'wb') as f:
                pickle.dump(self.metadatas, f)

            logger.info("✓ Database saved")
        except Exception as e:
            logger.error(f"Error saving database: {e}")

    def chunk_text(self, text: str, chunk_size: int = 400, overlap: int = 50) -> List[str]:
        """Split text into overlapping chunks"""
        words = text.split()
        chunks = []

        for i in range(0, len(words), chunk_size - overlap):
            chunk = ' '.join(words[i:i + chunk_size])
            if len(chunk.strip()) > 50:  # Minimum chunk size
                chunks.append(chunk)

        return chunks

    def add_plant(self, plant_name: str, spider_output: List[Dict]) -> int:
        """
        Add plant to knowledge base

        Args:
            plant_name: Name of the plant (e.g., "Rosa rubiginosa")
            spider_output: List of dicts with 'text' and 'metadata' keys

        Returns:
            Number of chunks added
        """

        if not spider_output:
            logger.warning(f"No data provided for {plant_name}")
            return 0

        total_chunks = 0
        new_embeddings = []
        new_documents = []
        new_metadatas = []

        for source in spider_output:
            text_content = source['text']
            metadata = source['metadata']

            # Add plant name to metadata
            metadata['plant_name'] = plant_name

            # Chunk the text
            chunks = self.chunk_text(text_content)

            if not chunks:
                continue

            # Generate embeddings
            logger.info(f"  Embedding {len(chunks)} chunks from {metadata.get('source', 'Unknown')}")
            embeddings = self.embedding_model.encode(
                chunks,
                show_progress_bar=False,
                convert_to_numpy=True
            )

            # Store each chunk
            for i, (chunk, embedding) in enumerate(zip(chunks, embeddings)):
                chunk_meta = metadata.copy()
                chunk_meta['chunk_id'] = i
                chunk_meta['total_chunks'] = len(chunks)

                new_embeddings.append(embedding)
                new_documents.append(chunk)
                new_metadatas.append(chunk_meta)

            total_chunks += len(chunks)
            logger.info(f"  ✓ {len(chunks)} chunks from {metadata['source']}")

        # Add to FAISS index
        if new_embeddings:
            embeddings_array = np.array(new_embeddings).astype('float32')
            self.index.add(embeddings_array)
            self.documents.extend(new_documents)
            self.metadatas.extend(new_metadatas)

            # Save to disk
            self._save_database()

            logger.info(f"🎉 Total: {total_chunks} chunks added for {plant_name}")

        return total_chunks

    def search(
        self,
        query: str,
        n_results: int = 5,
        plant_name: Optional[str] = None,
        min_relevance: float = 0.0
    ) -> Dict:
        """
        Search the knowledge base

        Args:
            query: Search query
            n_results: Number of results to return
            plant_name: Filter by specific plant
            min_relevance: Minimum relevance threshold (0-1)

        Returns:
            Dict with documents, metadatas, and relevance scores
        """

        if len(self.documents) == 0:
            return {
                "documents": [],
                "metadatas": [],
                "relevance_scores": [],
                "distances": []
            }

        # Generate query embedding
        query_embedding = self.embedding_model.encode([query])[0].astype('float32')

        # Search (get extra results for filtering)
        search_k = min(n_results * 3, len(self.documents))
        distances, indices = self.index.search(
            np.array([query_embedding]),
            search_k
        )

        # Filter and collect results
        filtered_docs = []
        filtered_metas = []
        filtered_distances = []
        filtered_relevance = []

        for dist, idx in zip(distances[0], indices[0]):
            if idx >= len(self.metadatas):
                continue

            meta = self.metadatas[idx]

            # Apply plant filter
            if plant_name and meta.get('plant_name') != plant_name:
                continue

            # Calculate relevance (inverse of distance)
            relevance = 1 / (1 + dist)

            # Apply relevance threshold
            if relevance < min_relevance:
                continue

            filtered_docs.append(self.documents[idx])
            filtered_metas.append(meta)
            filtered_distances.append(float(dist))
            filtered_relevance.append(float(relevance))

            if len(filtered_docs) >= n_results:
                break

        return {
            "documents": filtered_docs,
            "metadatas": filtered_metas,
            "distances": filtered_distances,
            "relevance_scores": filtered_relevance
        }

    def build_prompt(self, query: str, search_results: Dict) -> Optional[str]:
        """Build LLM prompt with retrieved context"""

        if not search_results['documents']:
            return None

        # Build context sections
        context_parts = []
        for doc, meta, rel in zip(
            search_results['documents'],
            search_results['metadatas'],
            search_results['relevance_scores']
        ):
            reliability = meta.get('reliability', 'unknown').upper()
            source = meta.get('source', 'Unknown')
            title = meta.get('title', 'Untitled')
            plant = meta.get('plant_name', 'Unknown')

            context_parts.append(
                f"[{reliability}] {source} - {title} "
                f"(Plant: {plant}, Relevance: {rel:.2f}):\n{doc}"
            )

        context = "\n\n".join(context_parts)

        prompt = f"""You are a botanical expert. Answer based on these scientific sources:

SOURCES:
{context}

QUESTION: {query}

INSTRUCTIONS:
- Answer ONLY using information above
- Cite sources (e.g., "According to Britannica...")
- If sources conflict, mention both viewpoints
- If answer not in sources, say so clearly
- Prioritize higher reliability and relevance sources

ANSWER:"""

        return prompt

    def ask(
        self,
        query: str,
        plant_name: Optional[str] = None,
        n_results: int = 5,
        min_relevance: float = 0.3
    ) -> Dict:
        """
        Complete RAG query

        Returns:
            Dict with prompt, sources, and search results
        """

        # Search
        results = self.search(
            query,
            n_results=n_results,
            plant_name=plant_name,
            min_relevance=min_relevance
        )

        if not results['documents']:
            return {
                "prompt": None,
                "sources": [],
                "answer": f"No relevant information found (searched {len(self.documents)} chunks)",
                "results": results
            }

        # Build prompt
        prompt = self.build_prompt(query, results)

        # Extract unique sources
        sources = []
        seen = set()
        for meta in results['metadatas']:
            key = (meta.get('source'), meta.get('url'))
            if key not in seen:
                sources.append({
                    "plant": meta.get('plant_name', 'Unknown'),
                    "source": meta.get('source', 'Unknown'),
                    "title": meta.get('title', 'Untitled'),
                    "url": meta.get('url', ''),
                    "reliability": meta.get('reliability', 'unknown')
                })
                seen.add(key)

        return {
            "prompt": prompt,
            "sources": sources,
            "answer": None,  # Fill this with LLM response
            "results": results
        }

    def stats(self) -> Dict:
        """Get database statistics"""

        if len(self.documents) == 0:
            return {
                "total_chunks": 0,
                "unique_plants": 0,
                "plants": [],
                "unique_sources": 0,
                "sources": []
            }

        plants = set()
        sources = set()

        for meta in self.metadatas:
            plants.add(meta.get('plant_name', 'Unknown'))
            sources.add(meta.get('source', 'Unknown'))

        return {
            "total_chunks": len(self.documents),
            "unique_plants": len(plants),
            "plants": sorted(list(plants)),
            "unique_sources": len(sources),
            "sources": sorted(list(sources))
        }

    def get_plant_info(self, plant_name: str) -> Dict:
        """Get all stored info about a specific plant"""

        plant_chunks = []
        plant_sources = set()

        for doc, meta in zip(self.documents, self.metadatas):
            if meta.get('plant_name') == plant_name:
                plant_chunks.append({
                    "text": doc,
                    "source": meta.get('source'),
                    "title": meta.get('title')
                })
                plant_sources.add(meta.get('source'))

        return {
            "plant": plant_name,
            "total_chunks": len(plant_chunks),
            "sources": sorted(list(plant_sources)),
            "chunks": plant_chunks
        }


# ============================================
# USAGE EXAMPLES
# ============================================

def example_usage():
    """How to use with your spider output"""

    # Your spider output (from the document you provided)
    spider_results = [
        {
            'text': 'sweetbrier, (Rosa eglanteria, orR. rubiginosa), small, prickly wildrose...',
            'metadata': {
                'source': 'Encyclopædia Britannica',
                'reliability': 'very_high',
                'url': 'https://www.britannica.com/plant/sweetbrier',
                'domain': 'www.britannica.com',
                'title': 'sweetbrier',
                'scraped_date': '2025-10-08',
                'content_type': 'ecological_information'
            }
        },
        # ... more sources
    ]

    # Initialize RAG
    rag = PlantRAGSystem()

    # Add plant
    chunks_added = rag.add_plant("Rosa rubiginosa", spider_results)
    print(f"Added {chunks_added} chunks")

    # Ask questions
    result = rag.ask("What does Rosa rubiginosa look like?")

    if result['prompt']:
        print("\n=== PROMPT FOR LLM ===")
        print(result['prompt'])
        print("\n=== SOURCES ===")
        for src in result['sources']:
            print(f"- [{src['reliability']}] {src['source']}: {src['title']}")

    # Database stats
    stats = rag.stats()
    print(f"\nDatabase: {stats['total_chunks']} chunks, {stats['unique_plants']} plants")


def load_from_file_example():
    """Load spider results from JSON file"""

    # If you saved your spider output to JSON:
    with open('rosa_rubiginosa_results.json', 'r') as f:
        spider_data = json.load(f)

    rag = PlantRAGSystem()
    rag.add_plant("Rosa rubiginosa", spider_data)

    # Query
    result = rag.ask("Is this plant fragrant?", min_relevance=0.4)
    print(result['prompt'])


# ============================================
# QUICK START
# ============================================

if __name__ == "__main__":

    # Quick test
    rag = PlantRAGSystem()

    # Check stats
    stats = rag.stats()
    print(f"Database contains: {stats['total_chunks']} chunks")
    print(f"Plants: {stats['plants']}")

    # Example query
    if stats['total_chunks'] > 0:
        result = rag.ask("What does this plant look like?")
        if result['prompt']:
            print("\n" + "="*60)
            print(result['prompt'][:500] + "...")
            print("="*60)
            print(result['answer'])

Database contains: 8 chunks
Plants: ['Rosa rubiginosa']

You are a botanical expert. Answer based on these scientific sources:

SOURCES:
[VERY_HIGH] Encyclopædia Britannica - sweetbrier (Plant: Rosa rubiginosa, Relevance: 0.46):
Our editors will review what you’ve submitted and determine whether to revise the article. sweetbrier, (Rosa eglanteria, orR. rubiginosa), small, prickly wildrosewith fragrant foliage and numerous smallpinkflowers. Native toEuropeand westernAsia, it is widely naturalized inNorth America, where it grows along roadsides and in p...
None


In [ ]:
main()


ENHANCED PLANT SPIDER RESULTS FOR: Rosa rubiginosa
Total sources collected: 6

1. NC State Extension Plants
   Title: Rosa America™'JACclam'
   Reliability: very_high
   Content length: cultivation_guide characters
   URL: dict_keys(['source', 'reliability', 'url', 'domain', 'title', 'scraped_date', 'content_type'])
2. NC State Extension Plants
   Title: Hibiscus rosa-sinensis
   Reliability: very_high
   Content length: botanical_reference characters
   URL: dict_keys(['source', 'reliability', 'url', 'domain', 'title', 'scraped_date', 'content_type'])
3. Encyclopædia Britannica
   Title: shrub rose
   Reliability: very_high
   Content length: general_information characters
   URL: dict_keys(['source', 'reliability', 'url', 'domain', 'title', 'scraped_date', 'content_type'])
4. Encyclopædia Britannica
   Title: sweetbrier
   Reliability: very_high
   Content length: ecological_information characters
   URL: dict_keys(['source', 'reliability', 'url', 'domain', 'title', 'scraped_date', 

[{'text': 'America is a climbing rose that was introduced in 1976 and was bred by William A. Warriner.\n\nThis is a vigorous climber that grows rapidly to 10-14 feet tall and 6-8 feet wide. It does best in full sun in fertile, moist, well-drained soil. For best results apply fertilizer in early spring and again in summer. Mulch to help retain moisture.\n\nAmerica begins blooming in spring and repeat blooms during the season. The fragrant flowers are double coral with a clove scent that contract nicely with the dark green foliage.\n\nUse this plant in vertical spaces along arbors or trellis, in borders, along walls or fences. It is even able to climb bare walls.\n\nInsects, Diseases, and Other Plant Problems:Has shown good resistance to powdery mildew, rust, and blackspot. See pests of roses to the left.\n\nThe Clemson Cooperative Extension Home and Garden\xa0Information Center has afactsheeton common rose diseases as well as afactsheeton insects and other related pests.\n\nVIDEO Create

In [ ]:
pip install git+https://github.com/Pleias/Pleias-Rag.git

  Cloning https://github.com/Pleias/Pleias-Rag.git to /tmp/pip-req-build-azvnzrii
  Running command git clone --filter=blob:none --quiet https://github.com/Pleias/Pleias-Rag.git /tmp/pip-req-build-azvnzrii
  Resolved https://github.com/Pleias/Pleias-Rag.git to commit a8ba0efd0bdb9bf8f8b71be1999cbdf00b9a1ce4
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 36.4/36.4 MB 27.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 438.2/438.2 MB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.0/180.0 kB 16.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.0/111.0 kB 10.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.4/45.4 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.9/3.9 MB 91.7 MB/s eta 0:00:00
  

In [ ]:
!git clone https://github.com/Pleias/Pleias-Rag.git

Cloning into 'Pleias-Rag'...
remote: Enumerating objects: 135, done.
remote: Counting objects: 100% (135/135), done.
remote: Compressing objects: 100% (80/80), done.
remote: Total 135 (delta 42), reused 119 (delta 33), pack-reused 0 (from 0)
Receiving objects: 100% (135/135), 46.37 KiB | 5.15 MiB/s, done.
Resolving deltas: 100% (42/42), done.


In [ ]:

from Pleias_Rag import RagSystem

# Initialize the RAG system with an optional model path
rag_system = RagSystem(
    search_type="vector",
    db_path="data/rag_system_db",
    embeddings_model="all-MiniLM-L6-v2",
    chunk_size=300,
    model_path="Qwen/Qwen3-0.6B"  # Optional - can also load model later
)

# Add documents to the system
rag_system.add_and_chunk_documents([
    "Neural networks are a type of machine learning model inspired by the human brain.",
    "GPT is a generative pre-trained transformer model developed by OpenAI.",
    "RAG (Retrieval-Augmented Generation) combines retrieval systems with language models."
])

# End-to-end RAG query
query = "What are neural networks?"
result = rag_system.query(query)

# Access the results
print(f"Query: {result['query']}")
print(f"Response: {result['response']}")

Loading model from Qwen/Qwen3-0.6B...
INFO 10-08 14:50:13 [utils.py:233] non-default args: {'trust_remote_code': True, 'disable_log_stats': True}


The argument `trust_remote_code` is to be used with Auto classes. It has no effect here and is ignored.


config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

INFO 10-08 14:50:45 [model.py:547] Resolved architecture: Qwen3ForCausalLM
WARNING 10-08 14:50:45 [model.py:1682] Your device 'Tesla T4' (with compute capability 7.5) doesn't support torch.bfloat16. Falling back to torch.float16 for compatibility.
WARNING 10-08 14:50:45 [model.py:1733] Casting torch.bfloat16 to torch.float16.
INFO 10-08 14:50:45 [model.py:1510] Using max model len 40960
INFO 10-08 14:50:45 [scheduler.py:205] Chunked prefill is enabled with max_num_batched_tokens=8192.


tokenizer_config.json: 0.00B [00:00, ?B/s]

...

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

INFO 10-08 14:52:38 [llm.py:306] Supported_tasks: ['generate']
Model loaded successfully


Adding requests:   0%|          | 0/1 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Query: What are neural networks?
Response: Analysis of the sources: <|source_analysis_end|>
<|source_analysis|>The sources are all about neural networks. The first source mentions that neural networks are a type of machine learning model inspired by the human brain. The second source states that GPT is a generative pre-trained transformer model developed by OpenAI. The third source explains that RAG combines retrieval systems with language models. All sources are about neural networks. The sources are all about neural networks. The sources are all about neural networks. The sources are all about neural networks. The sources are all about neural networks. The sources are all about neural networks. The sources are all about neural networks. The sources are all about neural networks. The sources are all about neural networks. The sources are all about neural networks. The sources are all about neural networks. The sources are all about neural networks. The sources are all about neural net

In [ ]:
import requests

url = "https://huggingface.co/ngxson/demo_simple_rag_py/resolve/main/cat-facts.txt"
response = requests.get(url)

# Save to local file
with open("cat-facts.txt", "wb") as f:
    f.write(response.content)

print("File downloaded successfully!")

File downloaded successfully!


In [ ]:
!pip install ollama

In [ ]:
import ollama


# Load the dataset

dataset = []
with open('cat-facts.txt', 'r') as file:
  dataset = file.readlines()
  print(f'Loaded {len(dataset)} entries')



# Implement the retrieval system

EMBEDDING_MODEL = 'hf.co/CompendiumLabs/bge-base-en-v1.5-gguf'
LANGUAGE_MODEL = 'hf.co/bartowski/Llama-3.2-1B-Instruct-GGUF'

# Each element in the VECTOR_DB will be a tuple (chunk, embedding)
# The embedding is a list of floats, for example: [0.1, 0.04, -0.34, 0.21, ...]
VECTOR_DB = []

def add_chunk_to_database(chunk):
  embedding = ollama.embed(model=EMBEDDING_MODEL, input=chunk)['embeddings'][0]
  VECTOR_DB.append((chunk, embedding))

for i, chunk in enumerate(dataset):
  add_chunk_to_database(chunk)
  print(f'Added chunk {i+1}/{len(dataset)} to the database')

def cosine_similarity(a, b):
  dot_product = sum([x * y for x, y in zip(a, b)])
  norm_a = sum([x ** 2 for x in a]) ** 0.5
  norm_b = sum([x ** 2 for x in b]) ** 0.5
  return dot_product / (norm_a * norm_b)

def retrieve(query, top_n=3):
  query_embedding = ollama.embed(model=EMBEDDING_MODEL, input=query)['embeddings'][0]
  # temporary list to store (chunk, similarity) pairs
  similarities = []
  for chunk, embedding in VECTOR_DB:
    similarity = cosine_similarity(query_embedding, embedding)
    similarities.append((chunk, similarity))
  # sort by similarity in descending order, because higher similarity means more relevant chunks
  similarities.sort(key=lambda x: x[1], reverse=True)
  # finally, return the top N most relevant chunks
  return similarities[:top_n]



# Chatbot

input_query = input('Ask me a question: ')
retrieved_knowledge = retrieve(input_query)

print('Retrieved knowledge:')
for chunk, similarity in retrieved_knowledge:
  print(f' - (similarity: {similarity:.2f}) {chunk}')

instruction_prompt = f'''You are a helpful chatbot.
Use only the following pieces of context to answer the question. Don't make up any new information:
{'\n'.join([f' - {chunk}' for chunk, similarity in retrieved_knowledge])}
'''
# print(instruction_prompt)

stream = ollama.chat(
  model=LANGUAGE_MODEL,
  messages=[
    {'role': 'system', 'content': instruction_prompt},
    {'role': 'user', 'content': input_query},
  ],
  stream=True,
)

# print the response from the chatbot in real-time
print('Chatbot response:')
for chunk in stream:
  print(chunk['message']['content'], end='', flush=True)

Loaded 150 entries


ConnectionError: Failed to connect to Ollama. Please check that Ollama is downloaded, running and accessible. https://ollama.com/download

In [ ]:
!pip install sentence-transformers chromadb==0.4.24

  Using cached chromadb-0.4.24-py3-none-any.whl.metadata (7.3 kB)
Using cached chromadb-0.4.24-py3-none-any.whl (525 kB)
  Attempting uninstall: chromadb
    Found existing installation: chromadb 1.1.1
    Uninstalling chromadb-1.1.1:
      Successfully uninstalled chromadb-1.1.1


In [ ]:
!pip install --upgrade chromadb

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.9/19.9 MB 119.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 105.4/105.4 kB 11.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 7.9 MB/s eta 0:00:00
  Attempting uninstall: posthog
    Found existing installation: posthog 6.7.6
    Uninstalling posthog-6.7.6:
      Successfully uninstalled posthog-6.7.6
  Attempting uninstall: chromadb
    Found existing installation: chromadb 0.4.22
    Uninstalling chromadb-0.4.22:
      Successfully uninstalled chromadb-0.4.22


In [ ]:

"""
Complete Plant Spider to RAG System Integration
Handles multiple plants and builds a persistent knowledge base
"""

import requests
from bs4 import BeautifulSoup
import time
from urllib.parse import urljoin, urlparse, quote_plus
from typing import List, Dict, Optional
import json
from dataclasses import dataclass
from datetime import datetime
import logging
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings

# Configure logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

# ============================================
# ENHANCED PLANT SPIDER (Your existing code)
# ============================================

@dataclass
class Source:
    text: str
    metadata: Dict
    url: str
    title: str
    reliability_score: float

class EnhancedPlantSpider:
    def __init__(self, delay: float = 1.5, max_sources: int = 10, articles_per_search: int = 3):
        self.delay = delay
        self.max_sources = max_sources
        self.articles_per_search = articles_per_search
        self.session = requests.Session()
        self.session.headers.update({
            'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'
        })

        self.search_sources = {
            'thespruce.com': {
                'reliability': 0.75,
                'search_url': 'https://www.thespruce.com/search?q={query}',
            },
            'extension.wisc.edu': {
                'reliability': 0.85,
                'search_url': 'https://hort.extension.wisc.edu/articles/?s={query}',
            },
            'britannica.com': {
                'reliability': 0.9,
                'search_url': 'https://www.britannica.com/search?query={query}',
            },
        }

        self.direct_sources = {
            'en.wikipedia.org': {'reliability': 0.95},
            'kew.org': {'reliability': 0.95},
        }

    def get_search_urls_for_plant(self, plant_name: str) -> List[str]:
        search_urls = []
        query = quote_plus(plant_name)

        for domain, config in self.search_sources.items():
            search_url = config['search_url'].format(query=query)
            search_urls.append(search_url)

        wiki_variations = [
            f"https://en.wikipedia.org/wiki/{plant_name.replace(' ', '_')}",
        ]
        search_urls.extend(wiki_variations)

        return search_urls

    def extract_article_links_from_search(self, search_url: str, plant_name: str) -> List[Dict[str, str]]:
        try:
            logger.info(f"Processing search page: {search_url}")
            response = self.session.get(search_url, timeout=15)
            response.raise_for_status()
            soup = BeautifulSoup(response.content, 'html.parser')

            # Simplified extraction - you can expand this
            articles = []
            for link in soup.find_all('a', href=True)[:20]:
                href = link.get('href')
                title = link.get_text(strip=True)

                if href and title and len(title) > 10:
                    if any(kw in href.lower() for kw in ['plant', 'flower', 'garden', 'wiki']):
                        full_url = urljoin(search_url, href)
                        articles.append({'url': full_url, 'title': title})

            return articles[:self.articles_per_search]
        except Exception as e:
            logger.error(f"Error: {e}")
            return []

    def extract_plant_info(self, url: str) -> Optional[Source]:
        try:
            response = self.session.get(url, timeout=15)
            response.raise_for_status()
            soup = BeautifulSoup(response.content, 'html.parser')

            # Remove unwanted elements
            for element in soup(['script', 'style', 'nav', 'header', 'footer']):
                element.decompose()

            # Extract title
            title = soup.find('h1')
            title = title.get_text(strip=True) if title else "Unknown"

            # Extract paragraphs
            paragraphs = soup.find_all('p')
            content = '\n\n'.join([p.get_text(strip=True) for p in paragraphs[:10]
                                  if len(p.get_text(strip=True)) > 50])

            if not content or len(content) < 100:
                return None

            domain = urlparse(url).netloc
            reliability_score = self.direct_sources.get(domain, {}).get('reliability', 0.7)

            metadata = {
                'source': domain.replace('www.', '').title(),
                'reliability': 'very_high' if reliability_score >= 0.9 else 'high',
                'url': url,
                'domain': domain,
                'title': title,
                'scraped_date': datetime.now().strftime('%Y-%m-%d'),
                'content_type': 'botanical_reference'
            }

            return Source(
                text=content,
                metadata=metadata,
                url=url,
                title=title,
                reliability_score=reliability_score
            )
        except Exception as e:
            logger.error(f"Error extracting from {url}: {e}")
            return None

    def collect_plant_sources(self, plant_name: str) -> List[Dict]:
        logger.info(f"Collecting sources for: {plant_name}")

        search_urls = self.get_search_urls_for_plant(plant_name)
        all_article_links = []
        sources = []
        processed_urls = set()

        # Get article links
        for search_url in search_urls[:3]:  # Limit to avoid too many requests
            if len(all_article_links) >= self.max_sources * 2:
                break
            article_links = self.extract_article_links_from_search(search_url, plant_name)
            all_article_links.extend(article_links)
            time.sleep(self.delay)

        # Extract content from articles
        for article in all_article_links[:self.max_sources]:
            url = article['url']

            if url in processed_urls:
                continue
            processed_urls.add(url)

            source = self.extract_plant_info(url)
            if source and len(source.text.strip()) > 150:
                sources.append({
                    "text": source.text,
                    "metadata": source.metadata
                })
                logger.info(f"✓ Extracted from {source.metadata['domain']}")

            time.sleep(self.delay)

        logger.info(f"Collected {len(sources)} sources for {plant_name}")
        return sources


# ============================================
# RAG SYSTEM INTEGRATION
# ============================================

class PlantRAGSystem:
    """
    RAG system that integrates with the spider
    Builds and maintains a persistent knowledge base
    """

    def __init__(self, db_path: str = "./plant_knowledge_db"):
        """Initialize RAG system"""
        self.embedding_model = SentenceTransformer('all-MiniLM-L6-v2')

        self.chroma_client = chromadb.Client(Settings(
            persist_directory=db_path,
            anonymized_telemetry=False
        ))

        self.collection = self.chroma_client.get_or_create_collection(
            name="plant_database",
            metadata={"description": "Botanical knowledge base"}
        )

        logger.info(f"RAG system initialized. Current items: {self.collection.count()}")

    def chunk_text(self, text: str, chunk_size: int = 400, overlap: int = 50) -> List[str]:
        """Split text into overlapping chunks"""
        words = text.split()
        chunks = []

        for i in range(0, len(words), chunk_size - overlap):
            chunk = ' '.join(words[i:i + chunk_size])
            if chunk.strip():
                chunks.append(chunk)

        return chunks

    def add_plant_to_database(self, plant_name: str, spider_results: List[Dict]):
        """
        Add a plant's information to the RAG database

        Args:
            plant_name: Name of the plant
            spider_results: Output from spider.collect_plant_sources()
        """

        if not spider_results:
            logger.warning(f"No sources to index for {plant_name}")
            return 0

        total_chunks = 0

        for result in spider_results:
            text_content = result['text']
            metadata = result['metadata']

            # Add plant name to metadata
            metadata['plant_name'] = plant_name

            # Chunk the text
            chunks = self.chunk_text(text_content)

            if not chunks:
                continue

            # Generate embeddings
            embeddings = self.embedding_model.encode(chunks).tolist()

            # Create unique IDs
            base_id = f"{plant_name}_{metadata['domain']}_{metadata['scraped_date']}"
            ids = [f"{base_id}_chunk_{i}" for i in range(len(chunks))]

            # Prepare metadata for each chunk
            metadatas = []
            for i in range(len(chunks)):
                chunk_meta = metadata.copy()
                chunk_meta['chunk_id'] = i
                chunk_meta['total_chunks'] = len(chunks)
                metadatas.append(chunk_meta)

            # Add to ChromaDB
            try:
                self.collection.add(
                    embeddings=embeddings,
                    documents=chunks,
                    ids=ids,
                    metadatas=metadatas
                )

                total_chunks += len(chunks)
                logger.info(f"✅ Indexed {len(chunks)} chunks from {metadata['source']}")
            except Exception as e:
                logger.error(f"Error indexing: {e}")

        logger.info(f"🎉 Added {total_chunks} chunks for {plant_name} from {len(spider_results)} sources")
        return total_chunks

    def search(self, query: str, n_results: int = 5,
               plant_name: Optional[str] = None,
               filter_reliability: Optional[str] = None) -> Dict:
        """
        Search the knowledge base

        Args:
            query: Search query
            n_results: Number of results
            plant_name: Optional - filter by specific plant
            filter_reliability: Optional - filter by reliability level
        """

        # Build filter
        where_filter = {}
        if plant_name:
            where_filter["plant_name"] = plant_name
        if filter_reliability:
            where_filter["reliability"] = filter_reliability

        # Search
        query_embedding = self.embedding_model.encode(query).tolist()

        results = self.collection.query(
            query_embeddings=[query_embedding],
            n_results=n_results,
            where=where_filter if where_filter else None
        )

        return results

    def build_rag_prompt(self, query: str, results: Dict) -> str:
        """Build prompt for LLM with retrieved context"""

        if not results['documents'][0]:
            return None

        context_sections = []
        for doc, meta in zip(results['documents'][0], results['metadatas'][0]):
            reliability = meta['reliability'].upper()
            source = meta['source']
            title = meta['title']
            plant = meta.get('plant_name', 'Unknown')

            context_sections.append(
                f"[{reliability}] {source} - {title} (Plant: {plant}):\n{doc}"
            )

        context = "\n\n".join(context_sections)

        prompt = f"""You are a botanical expert. Answer based on these scientific sources:

SOURCES:
{context}

QUESTION: {query}

INSTRUCTIONS:
- Answer ONLY using information above
- Cite sources (e.g., "According to Wikipedia...")
- If sources conflict, mention both
- If answer not in sources, say so clearly
- Prioritize higher reliability sources

ANSWER:"""

        return prompt

    def ask_question(self, query: str, plant_name: Optional[str] = None) -> Dict:
        """
        Complete RAG query

        Returns dict with prompt, sources, and raw results
        """

        results = self.search(query, plant_name=plant_name)

        if not results['documents'][0]:
            return {
                "answer": "No relevant information found.",
                "sources": [],
                "prompt": None
            }

        prompt = self.build_rag_prompt(query, results)

        # Extract unique sources
        sources = []
        seen = set()
        for meta in results['metadatas'][0]:
            key = (meta['source'], meta['url'])
            if key not in seen:
                sources.append({
                    "plant": meta.get('plant_name', 'Unknown'),
                    "source": meta['source'],
                    "title": meta['title'],
                    "url": meta['url'],
                    "reliability": meta['reliability']
                })
                seen.add(key)

        return {
            "prompt": prompt,
            "sources": sources,
            "raw_results": results
        }

    def get_database_stats(self) -> Dict:
        """Get statistics about the knowledge base"""

        total_items = self.collection.count()

        # Get all metadata to analyze
        if total_items == 0:
            return {"total_chunks": 0, "plants": [], "sources": []}

        all_data = self.collection.get()
        metadatas = all_data['metadatas']

        plants = set()
        sources = set()

        for meta in metadatas:
            plants.add(meta.get('plant_name', 'Unknown'))
            sources.add(meta.get('source', 'Unknown'))

        return {
            "total_chunks": total_items,
            "unique_plants": len(plants),
            "plants": sorted(list(plants)),
            "unique_sources": len(sources),
            "sources": sorted(list(sources))
        }


# ============================================
# INTEGRATED WORKFLOW
# ============================================

class IntegratedPlantSystem:
    """Complete system: Spider + RAG"""

    def __init__(self, spider_delay: float = 1.5, spider_max_sources: int = 6):
        self.spider = EnhancedPlantSpider(
            delay=spider_delay,
            max_sources=spider_max_sources
        )
        self.rag = PlantRAGSystem()

    def add_plant(self, plant_name: str) -> int:
        """
        Collect information about a plant and add to knowledge base

        Returns number of chunks indexed
        """

        logger.info(f"\n{'='*60}")
        logger.info(f"Processing: {plant_name}")
        logger.info(f"{'='*60}")

        # Collect sources using spider
        sources = self.spider.collect_plant_sources(plant_name)

        # Add to RAG database
        chunks_added = self.rag.add_plant_to_database(plant_name, sources)

        return chunks_added

    def add_multiple_plants(self, plant_names: List[str]):
        """Add multiple plants to the knowledge base"""

        results = {}

        for plant_name in plant_names:
            try:
                chunks = self.add_plant(plant_name)
                results[plant_name] = {"success": True, "chunks": chunks}
            except Exception as e:
                logger.error(f"Failed to process {plant_name}: {e}")
                results[plant_name] = {"success": False, "error": str(e)}

        return results

    def ask(self, question: str, plant_name: Optional[str] = None):
        """Ask a question about plants"""
        return self.rag.ask_question(question, plant_name=plant_name)

    def stats(self):
        """Get knowledge base statistics"""
        return self.rag.get_database_stats()


# ============================================
# USAGE EXAMPLES
# ============================================

def example_usage():
    """Demonstration of the complete system"""

    # Initialize the integrated system
    system = IntegratedPlantSystem(spider_delay=1.5, spider_max_sources=4)

    # Example 1: Add single plant
    print("\n=== Adding Rosa rubiginosa ===")
    chunks = system.add_plant("Rosa rubiginosa")
    print(f"Added {chunks} chunks to database")

    # Example 2: Add multiple plants
    print("\n=== Adding multiple plants ===")
    plants = ["Lavandula angustifolia", "Salvia officinalis"]
    results = system.add_multiple_plants(plants)

    for plant, result in results.items():
        if result['success']:
            print(f"✓ {plant}: {result['chunks']} chunks")
        else:
            print(f"✗ {plant}: {result['error']}")

    # Example 3: Check database stats
    print("\n=== Database Statistics ===")
    stats = system.stats()
    print(f"Total chunks: {stats['total_chunks']}")
    print(f"Plants: {', '.join(stats['plants'])}")
    print(f"Sources: {', '.join(stats['sources'])}")

    # Example 4: Ask questions
    print("\n=== Asking Questions ===")

    result = system.ask("What are the growing conditions for Rosa rubiginosa?")

    if result['prompt']:
        print("\nPROMPT FOR LLM:")
        print("="*60)
        print(result['prompt'])
        print("="*60)

        print("\nSOURCES USED:")
        for source in result['sources']:
            print(f"- [{source['reliability']}] {source['source']}: {source['title']}")

    # Example 5: Query specific plant
    result = system.ask(
        "Is this plant fragrant?",
        plant_name="Rosa rubiginosa"
    )


# ============================================
# MAIN ENTRY POINT
# ============================================

if __name__ == "__main__":

    # Quick start example
    system = IntegratedPlantSystem()

    # Add your plant
    system.add_plant("Rosa rubiginosa")

    # Ask questions
    result = system.ask("What does this plant look like?")

    if result['prompt']:
        print(result['prompt'])
        print("\n\nSources:", len(result['sources']))

ImportError: cannot import name 'maybe_cast_one_to_many_ids' from 'chromadb.api.types' (/usr/local/lib/python3.12/dist-packages/chromadb/api/types.py)

In [ ]:

!pip install faiss-cpu  # For CPU
# OR
!pip install faiss-gpu  # For GPU acceleration

!pip install sentence-transformers beautifulsoup4 requests

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.4/31.4 MB 41.9 MB/s eta 0:00:00
ERROR: Could not find a version that satisfies the requirement faiss-gpu (from versions: none)
ERROR: No matching distribution found for faiss-gpu


In [ ]:

"""
Complete Plant Spider to RAG System Integration with FAISS
Handles multiple plants and builds a persistent knowledge base
"""

import requests
from bs4 import BeautifulSoup
import time
from urllib.parse import urljoin, urlparse, quote_plus
from typing import List, Dict, Optional
import json
import pickle
from dataclasses import dataclass
from datetime import datetime
import logging
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np
from pathlib import Path

# Configure logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

# ============================================
# ENHANCED PLANT SPIDER
# ============================================

@dataclass
class Source:
    text: str
    metadata: Dict
    url: str
    title: str
    reliability_score: float

class EnhancedPlantSpider:
    def __init__(self, delay: float = 1.5, max_sources: int = 10, articles_per_search: int = 3):
        self.delay = delay
        self.max_sources = max_sources
        self.articles_per_search = articles_per_search
        self.session = requests.Session()
        self.session.headers.update({
            'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'
        })

        self.search_sources = {
            'thespruce.com': {
                'reliability': 0.75,
                'search_url': 'https://www.thespruce.com/search?q={query}',
            },
            'extension.wisc.edu': {
                'reliability': 0.85,
                'search_url': 'https://hort.extension.wisc.edu/articles/?s={query}',
            },
            'britannica.com': {
                'reliability': 0.9,
                'search_url': 'https://www.britannica.com/search?query={query}',
            },
        }

        self.direct_sources = {
            'en.wikipedia.org': {'reliability': 0.95},
            'kew.org': {'reliability': 0.95},
        }

    def get_search_urls_for_plant(self, plant_name: str) -> List[str]:
        search_urls = []
        query = quote_plus(plant_name)

        for domain, config in self.search_sources.items():
            search_url = config['search_url'].format(query=query)
            search_urls.append(search_url)

        wiki_variations = [
            f"https://en.wikipedia.org/wiki/{plant_name.replace(' ', '_')}",
        ]
        search_urls.extend(wiki_variations)

        return search_urls

    def extract_article_links_from_search(self, search_url: str, plant_name: str) -> List[Dict[str, str]]:
        try:
            logger.info(f"Processing search page: {search_url}")
            response = self.session.get(search_url, timeout=15)
            response.raise_for_status()
            soup = BeautifulSoup(response.content, 'html.parser')

            articles = []
            for link in soup.find_all('a', href=True)[:20]:
                href = link.get('href')
                title = link.get_text(strip=True)

                if href and title and len(title) > 10:
                    if any(kw in href.lower() for kw in ['plant', 'flower', 'garden', 'wiki']):
                        full_url = urljoin(search_url, href)
                        articles.append({'url': full_url, 'title': title})

            return articles[:self.articles_per_search]
        except Exception as e:
            logger.error(f"Error: {e}")
            return []

    def extract_plant_info(self, url: str) -> Optional[Source]:
        try:
            response = self.session.get(url, timeout=15)
            response.raise_for_status()
            soup = BeautifulSoup(response.content, 'html.parser')

            for element in soup(['script', 'style', 'nav', 'header', 'footer']):
                element.decompose()

            title = soup.find('h1')
            title = title.get_text(strip=True) if title else "Unknown"

            paragraphs = soup.find_all('p')
            content = '\n\n'.join([p.get_text(strip=True) for p in paragraphs[:10]
                                  if len(p.get_text(strip=True)) > 50])

            if not content or len(content) < 100:
                return None

            domain = urlparse(url).netloc
            reliability_score = self.direct_sources.get(domain, {}).get('reliability', 0.7)

            metadata = {
                'source': domain.replace('www.', '').title(),
                'reliability': 'very_high' if reliability_score >= 0.9 else 'high',
                'url': url,
                'domain': domain,
                'title': title,
                'scraped_date': datetime.now().strftime('%Y-%m-%d'),
                'content_type': 'botanical_reference'
            }

            return Source(
                text=content,
                metadata=metadata,
                url=url,
                title=title,
                reliability_score=reliability_score
            )
        except Exception as e:
            logger.error(f"Error extracting from {url}: {e}")
            return None

    def collect_plant_sources(self, plant_name: str) -> List[Dict]:
        logger.info(f"Collecting sources for: {plant_name}")

        search_urls = self.get_search_urls_for_plant(plant_name)
        all_article_links = []
        sources = []
        processed_urls = set()

        for search_url in search_urls[:3]:
            if len(all_article_links) >= self.max_sources * 2:
                break
            article_links = self.extract_article_links_from_search(search_url, plant_name)
            all_article_links.extend(article_links)
            time.sleep(self.delay)

        for article in all_article_links[:self.max_sources]:
            url = article['url']

            if url in processed_urls:
                continue
            processed_urls.add(url)

            source = self.extract_plant_info(url)
            if source and len(source.text.strip()) > 150:
                sources.append({
                    "text": source.text,
                    "metadata": source.metadata
                })
                logger.info(f"✓ Extracted from {source.metadata['domain']}")

            time.sleep(self.delay)

        logger.info(f"Collected {len(sources)} sources for {plant_name}")
        return sources


# ============================================
# FAISS-BASED RAG SYSTEM
# ============================================

class PlantRAGSystem:
    """
    RAG system using FAISS for vector storage
    More efficient and scalable than ChromaDB
    """

    def __init__(self, db_path: str = "./plant_knowledge_db"):
        """Initialize FAISS-based RAG system"""
        self.db_path = Path(db_path)
        self.db_path.mkdir(exist_ok=True)

        # Initialize embedding model
        logger.info("Loading embedding model...")
        self.embedding_model = SentenceTransformer('all-MiniLM-L6-v2')
        self.embedding_dim = 384  # Dimension for all-MiniLM-L6-v2

        # FAISS index
        self.index = None
        self.documents = []  # Store actual text chunks
        self.metadatas = []  # Store metadata for each chunk

        # File paths
        self.index_path = self.db_path / "faiss_index.bin"
        self.docs_path = self.db_path / "documents.pkl"
        self.meta_path = self.db_path / "metadata.pkl"

        # Load existing database if available
        self._load_database()

        logger.info(f"RAG system initialized. Current items: {len(self.documents)}")

    def _initialize_index(self):
        """Initialize FAISS index"""
        # Use IndexFlatL2 for exact search (can switch to IndexIVFFlat for speed)
        self.index = faiss.IndexFlatL2(self.embedding_dim)
        # Alternative for large datasets:
        # self.index = faiss.IndexIVFFlat(
        #     faiss.IndexFlatL2(self.embedding_dim),
        #     self.embedding_dim,
        #     100  # number of clusters
        # )

    def _load_database(self):
        """Load existing FAISS database"""
        if self.index_path.exists() and self.docs_path.exists() and self.meta_path.exists():
            try:
                self.index = faiss.read_index(str(self.index_path))

                with open(self.docs_path, 'rb') as f:
                    self.documents = pickle.load(f)

                with open(self.meta_path, 'rb') as f:
                    self.metadatas = pickle.load(f)

                logger.info(f"Loaded existing database with {len(self.documents)} chunks")
            except Exception as e:
                logger.warning(f"Error loading database: {e}. Starting fresh.")
                self._initialize_index()
        else:
            self._initialize_index()

    def _save_database(self):
        """Persist FAISS database to disk"""
        try:
            faiss.write_index(self.index, str(self.index_path))

            with open(self.docs_path, 'wb') as f:
                pickle.dump(self.documents, f)

            with open(self.meta_path, 'wb') as f:
                pickle.dump(self.metadatas, f)

            logger.info("Database saved successfully")
        except Exception as e:
            logger.error(f"Error saving database: {e}")

    def chunk_text(self, text: str, chunk_size: int = 400, overlap: int = 50) -> List[str]:
        """Split text into overlapping chunks"""
        words = text.split()
        chunks = []

        for i in range(0, len(words), chunk_size - overlap):
            chunk = ' '.join(words[i:i + chunk_size])
            if chunk.strip():
                chunks.append(chunk)

        return chunks

    def add_plant_to_database(self, plant_name: str, spider_results: List[Dict]):
        """
        Add a plant's information to the FAISS database

        Args:
            plant_name: Name of the plant
            spider_results: Output from spider.collect_plant_sources()
        """

        if not spider_results:
            logger.warning(f"No sources to index for {plant_name}")
            return 0

        total_chunks = 0
        new_embeddings = []
        new_documents = []
        new_metadatas = []

        for result in spider_results:
            text_content = result['text']
            metadata = result['metadata']

            # Add plant name to metadata
            metadata['plant_name'] = plant_name

            # Chunk the text
            chunks = self.chunk_text(text_content)

            if not chunks:
                continue

            # Generate embeddings
            embeddings = self.embedding_model.encode(chunks, show_progress_bar=False)

            # Prepare metadata for each chunk
            for i, (chunk, embedding) in enumerate(zip(chunks, embeddings)):
                chunk_meta = metadata.copy()
                chunk_meta['chunk_id'] = i
                chunk_meta['total_chunks'] = len(chunks)

                new_embeddings.append(embedding)
                new_documents.append(chunk)
                new_metadatas.append(chunk_meta)

            total_chunks += len(chunks)
            logger.info(f"✅ Indexed {len(chunks)} chunks from {metadata['source']}")

        # Add to FAISS index
        if new_embeddings:
            embeddings_array = np.array(new_embeddings).astype('float32')
            self.index.add(embeddings_array)
            self.documents.extend(new_documents)
            self.metadatas.extend(new_metadatas)

            # Save to disk
            self._save_database()

        logger.info(f"🎉 Added {total_chunks} chunks for {plant_name} from {len(spider_results)} sources")
        return total_chunks

    def search(self, query: str, n_results: int = 5,
               plant_name: Optional[str] = None,
               filter_reliability: Optional[str] = None) -> Dict:
        """
        Search the knowledge base using FAISS

        Args:
            query: Search query
            n_results: Number of results
            plant_name: Optional - filter by specific plant
            filter_reliability: Optional - filter by reliability level
        """

        if len(self.documents) == 0:
            return {"documents": [[]], "metadatas": [[]], "distances": [[]]}

        # Generate query embedding
        query_embedding = self.embedding_model.encode([query])[0].astype('float32')

        # Search FAISS index (get more results for filtering)
        search_k = min(n_results * 5, len(self.documents))
        distances, indices = self.index.search(
            np.array([query_embedding]),
            search_k
        )

        # Filter results
        filtered_docs = []
        filtered_metas = []
        filtered_distances = []

        for dist, idx in zip(distances[0], indices[0]):
            if idx >= len(self.metadatas):
                continue

            meta = self.metadatas[idx]

            # Apply filters
            if plant_name and meta.get('plant_name') != plant_name:
                continue
            if filter_reliability and meta.get('reliability') != filter_reliability:
                continue

            filtered_docs.append(self.documents[idx])
            filtered_metas.append(meta)
            filtered_distances.append(float(dist))

            if len(filtered_docs) >= n_results:
                break

        return {
            "documents": [filtered_docs],
            "metadatas": [filtered_metas],
            "distances": [filtered_distances]
        }

    def build_rag_prompt(self, query: str, results: Dict) -> str:
        """Build prompt for LLM with retrieved context"""

        if not results['documents'][0]:
            return None

        context_sections = []
        for doc, meta, dist in zip(results['documents'][0],
                                   results['metadatas'][0],
                                   results['distances'][0]):
            reliability = meta['reliability'].upper()
            source = meta['source']
            title = meta['title']
            plant = meta.get('plant_name', 'Unknown')

            context_sections.append(
                f"[{reliability}] {source} - {title} (Plant: {plant}, Relevance: {1/(1+dist):.2f}):\n{doc}"
            )

        context = "\n\n".join(context_sections)

        prompt = f"""You are a botanical expert. Answer based on these scientific sources:

SOURCES:
{context}

QUESTION: {query}

INSTRUCTIONS:
- Answer ONLY using information above
- Cite sources (e.g., "According to Wikipedia...")
- If sources conflict, mention both
- If answer not in sources, say so clearly
- Prioritize higher reliability sources

ANSWER:"""

        return prompt

    def ask_question(self, query: str, plant_name: Optional[str] = None) -> Dict:
        """
        Complete RAG query

        Returns dict with prompt, sources, and raw results
        """

        results = self.search(query, plant_name=plant_name)

        if not results['documents'][0]:
            return {
                "answer": "No relevant information found.",
                "sources": [],
                "prompt": None
            }

        prompt = self.build_rag_prompt(query, results)

        # Extract unique sources
        sources = []
        seen = set()
        for meta in results['metadatas'][0]:
            key = (meta['source'], meta['url'])
            if key not in seen:
                sources.append({
                    "plant": meta.get('plant_name', 'Unknown'),
                    "source": meta['source'],
                    "title": meta['title'],
                    "url": meta['url'],
                    "reliability": meta['reliability']
                })
                seen.add(key)

        return {
            "prompt": prompt,
            "sources": sources,
            "raw_results": results
        }

    def get_database_stats(self) -> Dict:
        """Get statistics about the knowledge base"""

        total_items = len(self.documents)

        if total_items == 0:
            return {"total_chunks": 0, "plants": [], "sources": []}

        plants = set()
        sources = set()

        for meta in self.metadatas:
            plants.add(meta.get('plant_name', 'Unknown'))
            sources.add(meta.get('source', 'Unknown'))

        return {
            "total_chunks": total_items,
            "unique_plants": len(plants),
            "plants": sorted(list(plants)),
            "unique_sources": len(sources),
            "sources": sorted(list(sources))
        }


# ============================================
# INTEGRATED WORKFLOW
# ============================================

class IntegratedPlantSystem:
    """Complete system: Spider + FAISS RAG"""

    def __init__(self, spider_delay: float = 1.5, spider_max_sources: int = 6):
        self.spider = EnhancedPlantSpider(
            delay=spider_delay,
            max_sources=spider_max_sources
        )
        self.rag = PlantRAGSystem()

    def add_plant(self, plant_name: str) -> int:
        """
        Collect information about a plant and add to knowledge base

        Returns number of chunks indexed
        """

        logger.info(f"\n{'='*60}")
        logger.info(f"Processing: {plant_name}")
        logger.info(f"{'='*60}")

        # Collect sources using spider
        sources = self.spider.collect_plant_sources(plant_name)

        # Add to RAG database
        chunks_added = self.rag.add_plant_to_database(plant_name, sources)

        return chunks_added

    def add_multiple_plants(self, plant_names: List[str]):
        """Add multiple plants to the knowledge base"""

        results = {}

        for plant_name in plant_names:
            try:
                chunks = self.add_plant(plant_name)
                results[plant_name] = {"success": True, "chunks": chunks}
            except Exception as e:
                logger.error(f"Failed to process {plant_name}: {e}")
                results[plant_name] = {"success": False, "error": str(e)}

        return results

    def ask(self, question: str, plant_name: Optional[str] = None):
        """Ask a question about plants"""
        return self.rag.ask_question(question, plant_name=plant_name)

    def stats(self):
        """Get knowledge base statistics"""
        return self.rag.get_database_stats()


# ============================================
# USAGE EXAMPLES
# ============================================

def example_usage():
    """Demonstration of the complete system"""

    # Initialize the integrated system
    system = IntegratedPlantSystem(spider_delay=1.5, spider_max_sources=4)

    # Example 1: Add single plant
    print("\n=== Adding Rosa rubiginosa ===")
    chunks = system.add_plant("Rosa rubiginosa")
    print(f"Added {chunks} chunks to database")

    # Example 2: Add multiple plants
    print("\n=== Adding multiple plants ===")
    plants = ["Lavandula angustifolia", "Salvia officinalis"]
    results = system.add_multiple_plants(plants)

    for plant, result in results.items():
        if result['success']:
            print(f"✓ {plant}: {result['chunks']} chunks")
        else:
            print(f"✗ {plant}: {result['error']}")

    # Example 3: Check database stats
    print("\n=== Database Statistics ===")
    stats = system.stats()
    print(f"Total chunks: {stats['total_chunks']}")
    print(f"Plants: {', '.join(stats['plants'])}")
    print(f"Sources: {', '.join(stats['sources'])}")

    # Example 4: Ask questions
    print("\n=== Asking Questions ===")

    result = system.ask("What are the growing conditions for Rosa rubiginosa?")

    if result['prompt']:
        print("\nPROMPT FOR LLM:")
        print("="*60)
        print(result['prompt'])
        print("="*60)

        print("\nSOURCES USED:")
        for source in result['sources']:
            print(f"- [{source['reliability']}] {source['source']}: {source['title']}")

    # Example 5: Query specific plant
    result = system.ask(
        "Is this plant fragrant?",
        plant_name="Rosa rubiginosa"
    )


# ============================================
# MAIN ENTRY POINT
# ============================================

if __name__ == "__main__":

    # Quick start example
    system = IntegratedPlantSystem()

    # Add your plant
    system.add_plant("Rosa rubiginosa")

    # Ask questions
    result = system.ask("What does this plant look like?")

    if result['prompt']:
        print(result['prompt'])
        print("\n\nSources:", len(result['sources']))

You are a botanical expert. Answer based on these scientific sources:

SOURCES:
[HIGH] Thespruce.Com - Introducing 'In the Weeds With Plant People': A New Series From The Spruce (Plant: Rosa rubiginosa, Relevance: 0.41):
closed during the day—gotta love a plant parent who prioritizes happy plants.

[HIGH] Thespruce.Com - Introducing 'In the Weeds With Plant People': A New Series From The Spruce (Plant: Rosa rubiginosa, Relevance: 0.41):
closed during the day—gotta love a plant parent who prioritizes happy plants.

[HIGH] Thespruce.Com - Introducing 'In the Weeds With Plant People': A New Series From The Spruce (Plant: Rosa rubiginosa, Relevance: 0.41):
Plant parents love plants. There’s no way around it. Personally, there’s not much that I enjoy more than taking care ofmy own plant collection. But, if I had to pick just one activity, it would be checking outotherpeople’s collections. I canscroll for hourson Instagram looking at pictures and videos of other people’s indoor jungles—it ju